# Demo chatbot có trích dẫn trên Colab L4 (job demo, tuần 7)

FastAPI (`app/api.py`, config B4 giống run test), mở ra ngoài bằng tunnel cloudflared. Giao diện React (`web/`) chạy ngoài Colab,
đặt `VITE_API_URL` = link tunnel; domain của frontend phải nằm trong `CORS_ORIGINS` (cell 2).
Đầu vào trên Drive `MyDrive/bv-rag/`: `processed/{VER}/`, `vectors/{VER}/` (đã đóng băng). Đầu ra: log hỏi đáp
`runs/demo/demo_<stamp>_log.jsonl` và bản chép kịch bản `runs/demo/demo_<stamp>_script.{md,jsonl}` trên Drive.

1. **Runtime → Change runtime type → GPU L4 → Save → Connect.** Cell 1 dừng nếu GPU không phải L4.
2. **Runtime → Run all.** Khởi động ~3–5 phút (cài Ollama, tải model, nạp bge-m3 + reranker, warm-up). Bật trước buổi bảo vệ ít nhất 10 phút.
3. Cell 7 in link API `https://….trycloudflare.com`: gán vào `VITE_API_URL` của frontend. Cell 8 chạy kịch bản 6 câu dev để kiểm trước khi trình bày.
4. Xong: cell 9 tắt runtime (Colab vẫn tính unit khi để mở).
Trước khi commit notebook: Edit → Clear all outputs.

## 1. Kiểm tra GPU (bắt buộc L4)

In [ ]:
import subprocess
GPU = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
print(GPU)
assert "L4" in GPU, f"GPU là {GPU}, không phải L4: đổi runtime trước mọi lần chạy chính thức (decision-log 2026-10-25)"

## 2. Tham số, mount Drive

In [ ]:
import os, time
VER = "v1"
BRANCH = "main"
DEMO_CONFIG = "configs/B4.yaml"   # cùng config với run test B4_v1_test_20261107-1335
STAMP = time.strftime("%Y%m%d-%H%M")
CORS_ORIGINS = "http://localhost:5173"   # domain frontend React, cách nhau bởi dấu phẩy

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/bv-rag"
DEMO_DIR = f"{DRIVE_DIR}/runs/demo"
os.makedirs(DEMO_DIR, exist_ok=True)
os.environ.update(DRIVE_DIR=DRIVE_DIR, QDRANT_URL=":memory:", OLLAMA_URL="http://localhost:11434",
                  PYTHONPATH="/content/bv-rag", DEMO_CONFIG=DEMO_CONFIG, DEMO_LOG=f"{DEMO_DIR}/demo_{STAMP}_log.jsonl",
                  CORS_ORIGINS=CORS_ORIGINS)
!ls {DRIVE_DIR}/processed/{VER} {DRIVE_DIR}/vectors/{VER}

## 3. Clone repo, cài thư viện (base + demo)

In [ ]:
!rm -rf /content/bv-rag && git clone -q -b {BRANCH} https://github.com/daisyngocha/bv-rag.git /content/bv-rag
%cd /content/bv-rag
!git branch --show-current && git log --oneline -1
!pip -q install -r requirements/base.txt -r requirements/demo.txt
assert os.path.exists("app/api.py"), f"Nhánh {BRANCH} chưa có app/api.py: kiểm tra BRANCH ở cell 2"

## 4. Kiểm tra kho theo manifest

In [ ]:
MANIFEST = f"data/manifests/{VER}.json"
assert os.path.exists(MANIFEST), f"{VER} chưa đóng băng: không chạy benchmark chính thức trên dữ liệu chưa đóng băng"
r = subprocess.run(["python", "scripts/freeze_data.py", "verify", VER, "--processed", f"{DRIVE_DIR}/processed/{VER}",
                    "--vectors", f"{DRIVE_DIR}/vectors/{VER}"], capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, f"Drive processed/vectors {VER} không khớp {MANIFEST}: không chạy tiếp"

## 5. Cài Ollama, bật server, tải model

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import time, requests
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); break
    except requests.RequestException:
        time.sleep(1)
!ollama pull qwen2.5:7b-instruct-q4_K_M
!grep -iE "inference compute|library=cuda" /content/ollama.log | head -3

## 6. Bật API
API nạp Qwen (warmup) trước bge-m3 và reranker, dựng Qdrant `:memory:`; chờ `/health` trả `ok` (tối đa 10 phút).
Lỗi: xem `/content/api.log`.

In [ ]:
import subprocess, requests
api = subprocess.Popen(["uvicorn", "app.api:app", "--host", "127.0.0.1", "--port", "8000"],
                       stdout=open("/content/api.log", "w"), stderr=subprocess.STDOUT)
t0 = time.time()
while time.time() - t0 < 600:
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).json().get("ok"):
            break
    except requests.RequestException:
        pass
    assert api.poll() is None, "API dừng khi khởi động: xem /content/api.log"
    time.sleep(3)
print(f"API sẵn sàng sau {time.time() - t0:.0f} s")
!grep -E "Demo sẵn sàng|LLM trên GPU" /content/api.log | tail -2
!ollama ps

## 7. Tunnel cloudflared (link công khai tạm thời)
Mở cổng API 8000 cho frontend React. Link đổi mỗi lần chạy, mất khi tắt runtime.

In [ ]:
import re
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                       stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        URL = m.group(0); break
    time.sleep(1)
print("VITE_API_URL =", URL)

## 8. Chạy kịch bản demo (6 câu dev) qua API
Kiểm hành vi trước khi trình bày (4 câu trả lời có trích dẫn, 2 câu ngoài phạm vi từ chối); bản chép Markdown làm dự phòng cùng video.

In [ ]:
!python scripts/demo_check.py --api http://127.0.0.1:8000 --out {DEMO_DIR}/demo_{STAMP}_script

## 9. Tắt runtime
Log demo đã nằm trên Drive (`runs/demo/`).

In [ ]:
from google.colab import runtime
if input("Tắt runtime? (y/n) ").strip().lower() == "y":
    runtime.unassign()